In [4]:
import psycopg2
import os
from dotenv import load_dotenv
load_dotenv()

conn =  psycopg2.connect(os.getenv("DATABASE_URL"))

In [5]:
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS songs (
    id SERIAL PRIMARY KEY,
    title TEXT NOT NULL,
    artist TEXT,
    duration FLOAT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS fingerprints (
    hash BIGINT NOT NULL,
    song_id INT NOT NULL,
    time_offset INT NOT NULL,
    FOREIGN KEY (song_id) REFERENCES songs(id) ON DELETE CASCADE
);
""")

cur.execute("""
CREATE INDEX IF NOT EXISTS idx_fingerprint_hash
ON fingerprints(hash);
""")

conn.commit()
cur.close()
conn.close()

In [27]:
from utils import generate_hashes_for_audio
hashes = generate_hashes_for_audio("data/misirlou.mp3") 

In [19]:
hashes

array([[2923536390,          1],
       [2923638790,          1],
       [2924797958,          1],
       ...,
       [1560739845,       2255],
       [1561014277,       2255],
       [1561636869,       2255]], shape=(19084, 2))

In [21]:
conn =  psycopg2.connect(os.getenv("DATABASE_URL"))
cur = conn.cursor()

cur.execute("""
    INSERT INTO songs (title, artist, duration)
    VALUES (%s, %s, %s)
    RETURNING id;
""", ("DaFunk", "Daft Punk", 52))

song_id = cur.fetchone()[0]

conn.commit()
cur.close()
conn.close()

In [22]:
from psycopg2.extras import execute_values

conn =  psycopg2.connect(os.getenv("DATABASE_URL"))
cur = conn.cursor()

data = [(int(h), int(song_id), int(offset)) for h, offset in hashes]

query = """
    INSERT INTO fingerprints (hash, song_id, time_offset)
    VALUES %s
"""

execute_values(cur, query, data, page_size=10000)

conn.commit()
cur.close()
conn.close()

In [28]:
conn = psycopg2.connect(os.getenv("DATABASE_URL"))
cur = conn.cursor()

query_hashes = hashes[:, 0].astype(int).tolist()

if query_hashes:
    cur.execute(
        """
        SELECT hash, song_id, time_offset
        FROM fingerprints
        WHERE hash = ANY(%s)
        """,
        (query_hashes,),
    )
    results = cur.fetchall()
else:
    results = []

cur.close()
conn.close()

results

[(277184517, 1, 9),
 (277377029, 1, 9),
 (277532677, 1, 9),
 (277975045, 1, 9),
 (277286918, 1, 9),
 (278351878, 1, 9),
 (277897222, 1, 9),
 (278614022, 1, 9),
 (277700625, 1, 9),
 (277786641, 1, 9),
 (88543237, 1, 10),
 (89608197, 1, 10),
 (89153541, 1, 10),
 (89870341, 1, 10),
 (88956944, 1, 10),
 (89042960, 1, 10),
 (88625168, 1, 10),
 (89182226, 1, 10),
 (89788434, 1, 10),
 (180817925, 1, 10),
 (181882885, 1, 10),
 (181428229, 1, 10),
 (182145029, 1, 10),
 (181231632, 1, 10),
 (181317648, 1, 10),
 (180899856, 1, 10),
 (181456914, 1, 10),
 (182063122, 1, 10),
 (788992005, 1, 10),
 (790056965, 1, 10),
 (789602309, 1, 10),
 (790319109, 1, 10),
 (789405712, 1, 10),
 (789491728, 1, 10),
 (789073936, 1, 10),
 (789630994, 1, 10),
 (790237202, 1, 10),
 (986537999, 1, 11),
 (986624015, 1, 11),
 (986206223, 1, 11),
 (986763281, 1, 11),
 (987369489, 1, 11),
 (684548111, 1, 11),
 (684634127, 1, 11),
 (684216335, 1, 11),
 (684773393, 1, 11),
 (685379601, 1, 11),
 (1661820943, 1, 11),
 (16619069

In [ ]:
from collections import defaultdict, Counter

db_map = defaultdict(list)
for h, song_id, offset in results:
    db_map[h].append((song_id, offset))

votes = defaultdict(list)

for h, q_offset in hashes:
    h = int(h)
    q_offset = int(q_offset)
    for song_id, db_offset in db_map.get(h, []):
        delta = db_offset - q_offset
        votes[song_id].append(delta)

best_song = None
best_score = 0

for song_id, deltas in votes.items():
    counter = Counter(deltas)
    _, count = counter.most_common(1)[0]

    if count > best_score:
        best_score = count
        best_song = song_id

best_title = None
if best_song is not None:
    conn = psycopg2.connect(os.getenv("DATABASE_URL"))
    cur = conn.cursor()
    cur.execute("SELECT title FROM songs WHERE id = %s", (best_song,))
    row = cur.fetchone()
    best_title = row[0] if row else None
    cur.close()
    conn.close()

{"song_id": best_song, "title": best_title, "score": best_score}

{'song_id': 1, 'title': 'Misrlou', 'score': 56806}